# Biography databases

Scholarly biographical dictionaries Cultura can be compared with: one biography downloaded from each (`building_scripts/dictionary_texts.py`).

In [1]:
import json
import sys
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts" / "5-annotations" / "building_scripts"))
from dictionary_texts import DB, DICTIONARIES, OUT

texts = json.loads(OUT.read_text(encoding="utf-8"))
connection = duckdb.connect(str(DB), read_only=True)
pids_in_cultura = {pid for (pid,) in connection.execute("SELECT DISTINCT pid FROM individual_identifier WHERE pid IN (SELECT unnest(?))",
                                                        [[d["pid"] for d in DICTIONARIES.values() if "pid" in d]]).fetchall()}
print(f"{len(DICTIONARIES)} dictionaries, {sum('text' in row for row in texts.values())} with a downloaded biography")

21 dictionaries, 14 with a downloaded biography


## Overview table

In [2]:
REGIONS = {
    "deutsche_biographie": "German-speaking world", "dbe": "Spain, colonial Latin America", "hls": "Switzerland",
    "oebl": "Austria-Hungary", "dib": "Ireland", "dwb": "Wales", "sbl": "Sweden", "nbl": "Norway", "dbl": "Denmark",
    "iranica": "Persia, Islamic East", "encykorea": "Korea", "dhbb": "Brazil", "dcb": "Canada", "adb": "Australia",
    "dnzb": "New Zealand", "sep": "Philosophers", "mactutor": "Mathematicians", "catholic_encyclopedia": "Global, religious",
    "pauly_wissowa": "Antiquity", "smith_dgrbm": "Antiquity", "britannica_1911": "Global, pre-1900",
}


def status(row):
    if "text" in row:
        return "downloaded"
    if "error" in row:
        return "identifier query failed"
    last = row["attempts"][-1] if row.get("attempts") else {}
    error = last.get("error", "")
    if "403" in error:
        return "blocked (403 Forbidden)"
    if error:
        return "server unreachable"
    return "no text in page (JavaScript, bot check or scan)"


def identifier(dictionary):
    if "pid" not in dictionary:
        return "Wikisource", "Wikidata"
    return dictionary["pid"], "Cultura" if dictionary["pid"] in pids_in_cultura else "Wikidata"


rows = []
for key, dictionary in DICTIONARIES.items():
    row = texts.get(key, {})
    first = (row.get("attempts") or [{}])[0]
    qid = row.get("qid") or first.get("qid")
    pid, source = identifier(dictionary)
    rows.append({
        "Dictionary": dictionary["name"],
        "Region": REGIONS[key],
        "Identifier": pid,
        "Identifiers from": source,
        "Identifiers": row.get("wikidata_items"),
        "In Cultura": row.get("cultura_candidates"),
        "Individual": f'<a href="https://www.wikidata.org/wiki/{qid}" target="_blank">{row.get("label") or qid}</a>' if qid else "",
        "Biography": f'<a href="{row.get("url") or first.get("url")}" target="_blank">link</a>' if row.get("url") or first.get("url") else "",
        "Characters": len(row["text"]) if "text" in row else None,
        "Status": status(row),
    })
overview = pd.DataFrame(rows).sort_values(["Status", "Dictionary"]).reset_index(drop=True)
assert overview["Dictionary"].is_unique
overview["Status"].value_counts()

Status
downloaded                                         14
no text in page (JavaScript, bot check or scan)     3
blocked (403 Forbidden)                             2
identifier query failed                             1
server unreachable                                  1
Name: count, dtype: int64

In [3]:
(overview.style
    .hide(axis="index")
    .format({"Identifiers": "{:,.0f}", "In Cultura": "{:,.0f}", "Characters": "{:,.0f}"}, na_rep="–")
    .map(lambda value: "color: #2e7d32" if value == "downloaded" else "color: #9e9e9e", subset=["Status"])
    .set_table_styles([
        {"selector": "th", "props": "text-align: left; font-weight: 600; border-bottom: 1px solid #ccc; padding: 6px 10px"},
        {"selector": "td", "props": "text-align: left; padding: 5px 10px; border-bottom: 1px solid #eee"},
    ]))

Dictionary,Region,Identifier,Identifiers from,Identifiers,In Cultura,Individual,Biography,Characters,Status
Dictionary of New Zealand Biography,New Zealand,P2745,Cultura,"3,130","2,965",Q7690812,link,–,blocked (403 Forbidden)
Historisches Lexikon der Schweiz,Switzerland,P902,Wikidata,"36,484","21,687",Q96413705,link,–,blocked (403 Forbidden)
Catholic Encyclopedia (1913),"Global, religious",P3241,Cultura,"5,532","5,207",Urban III,link,"6,293",downloaded
Dansk Biografisk Leksikon,Denmark,P8341,Wikidata,"10,501","9,702",Ferdinand Lindgreen,link,"4,928",downloaded
Deutsche Biographie (NDB/ADB),German-speaking world,P7902,Wikidata,"988,274","469,978",Kurt Hamer,link,867,downloaded
Dictionary of Canadian Biography,Canada,P2753,Cultura,"5,939","5,652",Albine Gadbois,link,"7,796",downloaded
Dictionary of Welsh Biography,Wales,P1648,Cultura,"4,608","4,000",David Price,link,"2,031",downloaded
"Encyclopaedia Britannica, 11th edition (1911)","Global, pre-1900",Wikisource,Wikidata,"36,950","13,614",Heinrich Graetz,link,"2,986",downloaded
Encyclopaedia Iranica,"Persia, Islamic East",P3021,Cultura,834,728,Roland de Mecquenem,link,"3,527",downloaded
Encyclopedia of Korean Culture,Korea,P9475,Wikidata,"14,822","2,425",Q12596301,link,"1,036",downloaded
